## Report
```
Two input DNA strings of equal length (up to 1 kbp) are transferred to the GPU as char arrays.
A CUDA kernel assigns one thread per position and each thread compares the corresponding characters from both strings
and performs a global atomicAdd on a single counter if they differ.
The result is copied back to the host and printed as the Hamming distance.
```

In [1]:
%%writefile counting_point_mutations.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define S_INPUT "GAGCCTACTAACGGGAT"
#define T_INPUT "CATCGTAATGACGGCCT"

__global__ void counting_point_mutations_kernel(const char *s, const char *t, unsigned long long *count, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        if (s[idx] != t[idx]) {
            atomicAdd(count, 1ULL);
        }
    }
}

void launch_hamm_kernel(const char *h_s, const char *h_t, int n)
{
    char *d_s = NULL;
    char *d_t = NULL;
    unsigned long long *d_count = NULL;

    cudaMalloc((void **)&d_s, n * sizeof(char));
    cudaMalloc((void **)&d_t, n * sizeof(char));
    cudaMalloc((void **)&d_count, sizeof(unsigned long long));

    cudaMemset(d_count, 0, sizeof(unsigned long long));
    cudaMemcpy(d_s, h_s, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_t, h_t, n * sizeof(char), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (n + block_size - 1) / block_size;

    counting_point_mutations_kernel<<<grid_size, block_size>>>(d_s, d_t, d_count, n);

    cudaDeviceSynchronize();

    unsigned long long h_count = 0;

    cudaMemcpy(&h_count, d_count, sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    printf("%llu\n", h_count);

    cudaFree(d_s);
    cudaFree(d_t);
    cudaFree(d_count);
}

int main(void)
{
    const char *s = S_INPUT;
    const char *t = T_INPUT;
    int n = (int)strlen(s);

    launch_hamm_kernel(s, t, n);

    return 0;
}

Writing counting_point_mutations.cu


In [2]:
!nvcc -arch=sm_75 counting_point_mutations.cu -o counting_point_mutations

In [3]:
!./counting_point_mutations

7
